In [ ]:
# import libraries
try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass

import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
# Lendo os arquivos TSV. Como eles não têm cabeçalho, nós damos os nomes 'label' e 'text'
train_data = pd.read_csv(train_file_path, sep='\t', header=None, names=['label', 'text'])
test_data = pd.read_csv(test_file_path, sep='\t', header=None, names=['label', 'text'])

# Convertendo as etiquetas de texto para números: 'ham' = 0, 'spam' = 1
train_labels = train_data['label'].map({'ham': 0, 'spam': 1}).values
train_text = train_data['text'].values

test_labels = test_data['label'].map({'ham': 0, 'spam': 1}).values
test_text = test_data['text'].values

In [ ]:
# --- CÉLULA 4: TREINAMENTO COM LSTM (MEMÓRIA DE TEXTO) ---
from tensorflow.keras import layers

vectorize_layer = layers.TextVectorization(
    max_tokens=7500,
    output_mode='int',
    output_sequence_length=50
)
vectorize_layer.adapt(train_text)

model = tf.keras.Sequential([
    vectorize_layer,
    layers.Embedding(
        input_dim=len(vectorize_layer.get_vocabulary()),
        output_dim=64,
        mask_zero=True),

    # A MÁGICA ESTÁ AQUI: O LSTM entende a ordem das palavras e o contexto!
    layers.LSTM(64),

    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Voltamos para 10 épocas, pois o LSTM aprende muito mais rápido e melhor
print("Iniciando o treinamento com LSTM...")
history = model.fit(
    train_text,
    train_labels,
    epochs=10,
    validation_data=(test_text, test_labels),
    verbose=1
)
print("Treinamento finalizado!")

In [ ]:
def predict_message(pred_text):
    # Pede para o modelo prever a mensagem. O modelo espera uma lista, então passamos [pred_text]
    # O [0][0] no final serve para extrair o número puro da matriz que o Keras devolve
    pred_prob = model.predict(tf.constant([pred_text], dtype=tf.string))[0][0]

    # Se a probabilidade for maior ou igual a 50% (0.5), é spam. Senão, é ham.
    if pred_prob >= 0.5:
        label = "spam"
    else:
        label = "ham"

    # Retorna exatamente no formato que o teste exige
    prediction = [pred_prob, label]

    return prediction

# Teste manual
pred_text = "how are you doing today?"
prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
